# TherMAM-NeRF v2.9 3D Visualizer
This notebook loads the trained `thermamnerf_best.pth` from `v2.9.py`, extracts the geometry using Marching Cubes, and displays an interactive 3D thermal model using Plotly.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import tifffile
from PIL import Image
from skimage.measure import marching_cubes
import plotly.graph_objects as go
import math
from pathlib import Path
import random
from scipy.ndimage import gaussian_filter, map_coordinates

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

CFG = {
    'img_size'        : 128,
    'n_views'         : 5,
    'view_angles_deg' : [-90, -45, 0, 45, 90],
    'view_names'      : ['RL', 'RO', 'F', 'LO', 'LL'],
    'feat_channels'   : 32,
    'pos_enc_L'       : 8,
    'mlp_hidden'      : 256,
    'mlp_layers'      : 4,
    'mc_threshold'    : 0.3,
    'mc_resolution'   : 128,
}

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, out_channels: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2,  16, 3, padding=1), nn.GroupNorm(4, 16), nn.ReLU(inplace=False),
            nn.Conv2d(16, 32, 3, padding=1), nn.GroupNorm(8, 32), nn.ReLU(inplace=False),
            nn.Conv2d(32, 32, 3, padding=1), nn.GroupNorm(8, 32), nn.ReLU(inplace=False),
            nn.Conv2d(32, out_channels, 1),
        )

    def forward(self, tiff_norm: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = torch.stack([tiff_norm, mask], dim=1)
        out = self.net(x)
        return out * mask.unsqueeze(1)

class ThermamNeRFMLP(nn.Module):
    def __init__(self, pos_enc_dim: int, feat_dim: int, hidden: int = 256, n_layers: int = 6):
        super().__init__()
        in_dim  = pos_enc_dim + feat_dim
        layers  = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=False)]
        self.layers  = nn.ModuleList()
        self.skip_at = n_layers // 2 - 1
        prev = in_dim
        for i in range(n_layers - 1):
            if i == self.skip_at:
                self.layers.append(nn.Linear(prev + in_dim, hidden))
            else:
                self.layers.append(nn.Linear(prev, hidden))
            prev = hidden
        self.sigma_head = nn.Linear(hidden, 1)
        self.temp_head  = nn.Linear(hidden, 1)

    def forward(self, pe: torch.Tensor, feat: torch.Tensor) -> tuple:
        chunk_size = 131072
        if pe.shape[1] <= chunk_size:
            x0 = torch.cat([pe, feat], dim=-1)
            h  = x0
            for i, layer in enumerate(self.layers):
                if i == self.skip_at:
                    h = torch.cat([h, x0], dim=-1)
                h = F.relu(layer(h), inplace=False)
            sigma  = F.softplus(self.sigma_head(h))
            T_norm = torch.sigmoid(self.temp_head(h))
            return sigma, T_norm
            
        sigma_list, temp_list = [], []
        for i in range(0, pe.shape[1], chunk_size):
            pe_c = pe[:, i:i+chunk_size]
            feat_c = feat[:, i:i+chunk_size]
            x0 = torch.cat([pe_c, feat_c], dim=-1)
            h  = x0
            for j, layer in enumerate(self.layers):
                if j == self.skip_at:
                    h = torch.cat([h, x0], dim=-1)
                h = F.relu(layer(h), inplace=False)
            sigma_list.append(F.softplus(self.sigma_head(h)))
            temp_list.append(torch.sigmoid(self.temp_head(h)))
        return torch.cat(sigma_list, dim=1), torch.cat(temp_list, dim=1)

def positional_encoding(x, L=6, alpha=None):
    if alpha is None: alpha = float(L)
    pe = [x]
    for i in range(L):
        freq = 2.0 ** i
        w = 1.0
        if i > alpha: w = 0.0
        elif i > alpha - 1: w = alpha - i
        pe.append(torch.sin(x * math.pi * freq) * w)
        pe.append(torch.cos(x * math.pi * freq) * w)
    return torch.cat(pe, dim=-1)

def project_and_sample(pts, feat_maps, view_angles_rad):
    V = feat_maps.shape[1]
    sampled_feats = []
    for v in range(V):
        theta = view_angles_rad[v]
        c, s = torch.cos(theta), torch.sin(theta)
        xr = c * pts[..., 0] - s * pts[..., 2]
        yr = pts[..., 1]
        grid = torch.stack([xr, yr], dim=-1).unsqueeze(1)
        sf = F.grid_sample(feat_maps[:, v], grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        sampled_feats.append(sf.squeeze(2).transpose(1, 2))
    return torch.cat(sampled_feats, dim=-1)

def extract_3d_volume(encoder, mlp, tiffs_norm, masks, cfg, device, resolution=128, chunk=8192):
    R = resolution
    linspace = torch.linspace(-1, 1, R, device=device)
    zz, yy, xx = torch.meshgrid(linspace, linspace, linspace, indexing='ij')
    pts = torch.stack([xx, yy, zz], dim=-1).reshape(1, -1, 3)
    view_angles_rad = torch.tensor([math.radians(a) for a in cfg['view_angles_deg']], device=device)
    
    feat_maps = []
    for v in range(cfg['n_views']): 
        fm = encoder(tiffs_norm[:, v], masks[:, v])
        feat_maps.append(fm)
    feat_maps = torch.stack(feat_maps, dim=1)
    
    alpha_final = float(cfg['pos_enc_L'])
    sigma_all, T_all = [], []
    n_pts = pts.shape[1]
    
    with torch.no_grad():
        for i in range(0, n_pts, chunk):
            p    = pts[:, i:i+chunk]
            pe   = positional_encoding(p, L=cfg['pos_enc_L'], alpha=alpha_final)
            feat = project_and_sample(p, feat_maps, view_angles_rad)
            sg, tp = mlp(pe, feat)
            sigma_all.append(sg.squeeze().cpu())
            T_all.append(tp.squeeze().cpu())
            
    sigma_grid = torch.cat(sigma_all).reshape(R, R, R).numpy()
    T_grid     = torch.cat(T_all).reshape(R, R, R).numpy()
    return sigma_grid, T_grid

In [ ]:
TIFF_DIR = Path('../images')
UNET_DIR = Path('../masks')

def get_view_key(filename):
    for vk in CFG['view_names']:
        if f'_{vk}_' in filename or f'_{vk}.' in filename: return vk
    return None

def normalize_thermal(img):
    tmin, tmax = img.min(), img.max()
    if tmax - tmin < 1e-6:
        return np.zeros_like(img), tmin, tmax
    return (img - tmin) / (tmax - tmin), tmin, tmax

pd_ = {}
for tp in TIFF_DIR.rglob('*.tiff'):
    parts = tp.relative_to(TIFF_DIR).parts
    if len(parts) < 2: continue
    pid, lab, fn = parts[0], parts[1], parts[-1]
    vk = get_view_key(fn)
    if not vk: continue
    key = (pid, lab)
    if key not in pd_: pd_[key] = {'tiffs': {}, 'masks': {}}
    pd_[key]['tiffs'][vk] = tp

for mp in UNET_DIR.rglob('*.png'):
    parts = mp.relative_to(UNET_DIR).parts
    if len(parts) < 2: continue
    pid, lab, fn = parts[0], parts[1], parts[-1]
    vk = get_view_key(fn)
    if not vk: continue
    key = (pid, lab)
    if key in pd_: pd_[key]['masks'][vk] = mp

patients = []
for (pid, lab), d in pd_.items():
    if len(d['tiffs']) == 5 and len(d['masks']) == 5:
        patients.append({'id': pid, 'tiffs': d['tiffs'], 'masks': d['masks']})

patients.sort(key=lambda p: int(p['id'].split('_')[-1]) if '_' in p['id'] else p['id'])
print(f"Found {len(patients)} total patients in the dataset.")

def load_tiff_celsius(filepath, size):
    img = tifffile.imread(filepath)
    if img.ndim == 3: img = img[:, :, 0]
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    pil_img = Image.fromarray(img)
    pil_img = pil_img.resize((size, size), Image.Resampling.BILINEAR)
    return np.array(pil_img, dtype=np.float32)

def load_mask(filepath, size):
    img = Image.open(filepath).convert('L')
    img = img.resize((size, size), Image.Resampling.NEAREST)
    arr = np.array(img, dtype=np.float32) / 255.0
    return (arr > 0.5).astype(np.float32)


In [ ]:
# ════════════════════════════════════════════════════════════════
# 3D VISUALIZATION OF RANDOM PATIENTS (LAPTOP LOCAL) - UPRIGHT
# ════════════════════════════════════════════════════════════════
def visualize_random_patients_local(n=5):
    print("Loading checkpoints...")
    enc = SiameseEncoder(out_channels=CFG['feat_channels']).to(DEVICE)
    pos_ch = 3 * (2 * CFG['pos_enc_L'] + 1)
    mlp = ThermamNeRFMLP(pos_enc_dim=pos_ch, feat_dim=CFG['feat_channels'] * CFG['n_views'], hidden=CFG['mlp_hidden'], n_layers=CFG['mlp_layers']).to(DEVICE)

    ckpt_path = Path('thermamnerf_outputs2.9/thermamnerf_best.pth')
    if not ckpt_path.exists():
        print(f"Error: No trained model found at {ckpt_path}.")
        return

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    enc.load_state_dict({k.replace('module.', ''): v for k, v in ckpt['encoder'].items()})
    mlp.load_state_dict({k.replace('module.', ''): v for k, v in ckpt['mlp'].items()})
    enc.eval()
    mlp.eval()
    
    if len(patients) == 0:
        print("No patients found.")
        return

    selected = random.sample(patients, min(n, len(patients)))
    print(f"Generating 3D models for {len(selected)} patients...")

    for patient in selected:
        pid = patient['id']
        tiffs_norm, tiffs_abs, masks = [], [], []
        tmins, tmaxs = [], []
        for v in CFG['view_names']:
            raw = load_tiff_celsius(str(patient['tiffs'][v]), CFG['img_size'])
            normd, tmin, tmax = normalize_thermal(raw)
            mask = load_mask(str(patient['masks'][v]), CFG['img_size'])
            tiffs_norm.append(normd)
            tiffs_abs.append(raw)
            masks.append(mask)
            tmins.append(tmin)
            tmaxs.append(tmax)

        tiffs_norm_t = torch.tensor(np.stack(tiffs_norm), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        masks_t      = torch.tensor(np.stack(masks), dtype=torch.float32).unsqueeze(0).to(DEVICE)

        sigma_grid, T_grid = extract_3d_volume(enc, mlp, tiffs_norm_t, masks_t, CFG, DEVICE)
        # Use light gaussian filter to slightly soften the blocky overfitting geometry
        sigma_grid = gaussian_filter(sigma_grid, sigma=1.5)
        
        try:
            # Extract mesh directly from the 3D density grid
            verts, faces, normals, values = marching_cubes(sigma_grid, level=CFG['mc_threshold'])
            
            # FAST VECTORIZED TEMPERATURE MAPPING:
            # Verts returned by marching_cubes are exact spatial indices into the T_grid array
            x_coords = verts[:, 0]
            y_coords = verts[:, 1]
            z_coords = verts[:, 2]
            
            temps_norm = map_coordinates(T_grid, [x_coords, y_coords, z_coords], order=1, mode='nearest')
            
            mean_tmin = np.mean(tmins)
            mean_tmax = np.mean(tmaxs)
            temps_abs = temps_norm * (mean_tmax - mean_tmin) + mean_tmin
            
            fig = go.Figure(data=[
                go.Mesh3d(
                    x=verts[:, 2],     # Width mapped to X (left/right)
                    y=verts[:, 0],     # Depth mapped to Y (front/back)
                    z=-verts[:, 1],    # Height mapped to Z (inverted so head points up)
                    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                    colorscale='Jet',  # Use 'Jet' exactly like the old breastnet cell
                    intensity=temps_abs,  
                    colorbar_title='Temperature (°C)',
                    showscale=True
                )
            ])
            
            fig.update_layout(
                title=f"Patient: {pid} (3D Thermal Mesh - NeRF v2.9)",
                scene=dict(
                    xaxis_title='Width (X)', 
                    yaxis_title='Depth (Y)', 
                    zaxis_title='Height (Z)',
                    aspectmode='data',
                    camera=dict(up=dict(x=0, y=0, z=1), eye=dict(x=0, y=1.5, z=0.2))
                ),
                margin=dict(l=0, r=0, b=0, t=40)
            )
            fig.show()
        except ValueError:
            print(f"Could not generate 3D mesh for {pid} (volume might be empty).")

# Run it!
visualize_random_patients_local(n=3)
